<a href="https://colab.research.google.com/github/Innovatewithapple/CNNProjects/blob/main/PytorchPreTrainedCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import torch.nn as nn

from torchvision import datasets,transforms
from torch.utils.data import DataLoader
import torch.optim as optimizer

import timm
import os
from google.colab import userdata

In [12]:
# 'tf_' refers to the official weights ported from the Google/TensorFlow team
model = timm.create_model('tf_efficientnetv2_s', pretrained=True, num_classes=1)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

In [14]:
# 1. Get the config from your model (run this once)
config = model.default_cfg
print(config)

{'url': 'https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-effv2-weights/tf_efficientnetv2_s_21ft1k-d7dafa41.pth', 'hf_hub_id': 'timm/tf_efficientnetv2_s.in21k_ft_in1k', 'architecture': 'tf_efficientnetv2_s', 'tag': 'in21k_ft_in1k', 'custom_load': False, 'input_size': (3, 300, 300), 'test_input_size': (3, 384, 384), 'fixed_input_size': False, 'interpolation': 'bicubic', 'crop_pct': 1.0, 'crop_mode': 'center', 'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5), 'num_classes': 1000, 'pool_size': (10, 10), 'first_conv': 'conv_stem', 'classifier': 'classifier', 'license': 'apache-2.0'}


In [5]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print('key and username activated!')

key and username activated!


In [6]:
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:45<00:00, 89.0MB/s]



In [7]:
!unzip -q 140k-real-and-fake-faces.zip -d ./dataset_folder

In [8]:
train_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/train'
test_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/test'
validation_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/valid'

In [15]:
train_transfom = transforms.Compose([
    transforms.Resize((384,384)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    # We grab the 'mean' and 'std' keys directly from the model's brain!
    transforms.Normalize(mean=config['mean'],
                         std=config['std'])
])

val_transfom = transforms.Compose([
    transforms.Resize((384,384)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    # We grab the 'mean' and 'std' keys directly from the model's brain!
    transforms.Normalize(mean=config['mean'],
                         std=config['std'])
])

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
pin_memory = (True if torch.cuda.is_available() else False)

cpu


In [18]:
train_dataset = datasets.ImageFolder(root=train_dir,transform=train_transfom)
val_dataset = datasets.ImageFolder(root=validation_dir,transform=val_transfom)

train_loader = DataLoader(dataset=train_dataset,batch_size=32,shuffle=True,num_workers=2,pin_memory=pin_memory)
val_loader = DataLoader(dataset=val_dataset,batch_size=32,num_workers=2,pin_memory=pin_memory)

In [20]:
#Model initialisation
# 2. FREEZE THE BRAIN: Lock the ImageNet knowledge
for params in model.parameters():
  params.requires_grad = False

# 3. THE HEAD: This is your custom part (exactly like your image)
my_custom_head = nn.Sequential(
    nn.LazyLinear(128),
    nn.ReLU(),
    nn.Linear(128,1)
)

#Assenmble
assembled_Model = nn.Sequential(
    model,
    my_custom_head
)

model = model.to(device)

In [21]:
dummyImg = torch.randn(1,3,384,384).to(device)
model(dummyImg)
print('✅ Model Assembled! Ready for Epochs.')

✅ Model Assembled! Ready for Epochs.


In [22]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optimizer.Adam(model.parameters(),lr=0.001)

In [ ]:
#Training
epochs = 10

for epoch in range(epochs):
  model.train()
  train_loss = 0
  train_correct = 0
  train_total = 0

  for i, (images,labels) in enumerate(train_loader):
    images = images.to(device)
    labels = labels.float().unsqueeze(1).to(device)

    optimizer.zero_grad()

    output = model(images)
    loss = loss_fn(output,labels)

    loss.backward()
    optimizer.step()

    if i % 100 == 0:
        print(f"Batch {i} - Loss: {loss.item():.4f}", end='\r')

    train_loss += loss.item()
    preds = (torch.sigmoid(output) > 0.5).int()
    train_correct += (preds.squeeze() == labels.squeeze).sum().item()

    train_total += labels.size(0)

  training_accuracy = train_correct / train_total

  #evlauation
  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0

  with torch.no_grad():
    for images,labels in val_loader:
     images = images.to(device)
     labels = labels.float().unsqueeze(1).to(device)

     output = model(images)
     loss = loss_fn(output,labels)

     val_loss += loss.item()
     preds = (torch.sigmoid(output) > 0.5).int()
     val_correct += (preds.squeeze() == labels.squeeze).sum().item()

     val_total += labels.size(0)

  validation_accuracy = val_correct / val_total

      # Calculate averages for the whole epoch
  avg_train_loss = train_loss / len(train_loader)
  avg_val_loss = val_loss / len(val_loader)

  # The "Professional Dashboard" Print
  print(f"\n🚀 Epoch [{epoch+1}/{epochs}]")
  print(f"-------------------------------")
  print(f"TRAIN | Loss: {avg_train_loss:.4f} | Acc: {training_accuracy:.4f}")
  print(f"VALID | Loss: {avg_val_loss:.4f} | Acc: {validation_accuracy:.4f}")
  print(f"-------------------------------\n")
